# Chapter 3.1 Toy Flow Matching from Scratch

This split isolates the toy dataset. The goal is to see conditional paths, local velocity targets, learned marginal velocity, and the CFM object hierarchy before touching EB single-cell data.

## Tutorial setup

The setup cells keep only reusable infrastructure here: project-root discovery, deterministic seeds, output directories, and save/display helpers. The scientific objects are introduced in the experiment sections below.

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

try:
    import torch
except ImportError as exc:
    raise ImportError("This notebook requires PyTorch for CFM training and sampling.") from exc

CWD = Path.cwd().resolve()
root_candidates = [
    CWD,
    CWD.parent,
    CWD.parent.parent,
    CWD / "flow_matching_for_dynamic_biology",
    CWD.parent / "flow_matching_for_dynamic_biology",
    CWD.parent.parent / "flow_matching_for_dynamic_biology",
]
for candidate in root_candidates:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate.resolve()
        break
else:
    raise FileNotFoundError("Could not locate project root containing src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ch03_tutorial as ch03
from src.utils import set_seed
from src.data import load_eb_timecourse_for_ch03, copy_trajectorynet_eb_to_data
from src.models import VelocityMLP, count_parameters
from src.losses import cfm_batch, cfm_loss_from_pairs
from src.train import train_cfm_steps
from src.sampling import euler_sample, midpoint_sample, odeint_sample
from src.metrics import endpoint_metrics, mmd_rbf, sliced_wasserstein_distance, trajectory_straightness
from src.flow_matching_reporting import plot_nfe_vs_error

In [ ]:
SEED = int(os.environ.get("CH03_SEED", "42"))
set_seed(SEED)
rng = np.random.default_rng(SEED)

QUICK_MODE = os.environ.get("CH03_QUICK", "1") == "1"
SMOKE_MODE = os.environ.get("CH03_SMOKE_MODE", "0") == "1"
PAPER_FIGURE_MODE = os.environ.get("CH03_PAPER_FIGURE_MODE", "1") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

FULL_RUN_HINTS = {
    "eb_train_steps": 10000,
    "cnf_steps": 600,
    "time_ablation_steps": 1200,
    "capacity_steps": 1200,
    "notes": "Set CH03_QUICK=0 for full-run settings; default remains quick and reproducible.",
}

context = ch03.make_ch03_context(PROJECT_ROOT)
FIG_DIR = context.fig_dir
TABLE_DIR = context.table_dir
OUT_DIR = context.output_dir
PAPER_COLORS = ch03.PAPER_COLORS

figures_written = []
paper_ready_png_written = []
paper_ready_pdf_written = []
tables_written = []
paper_tables_written = []
skipped_items = []

set_paper_style = ch03.set_paper_style
add_panel_label = ch03.add_panel_label
short_strategy_label = ch03.short_strategy_label
clean_spines = ch03.clean_spines
format_metric_axis = ch03.format_metric_axis
add_note = ch03.add_note
as_np = ch03.as_np
subsample_idx = ch03.subsample_idx
robust_limits = ch03.robust_limits
format_axis = ch03.format_axis

set_paper_style()
print({"project_root": str(PROJECT_ROOT), "device": DEVICE, "quick_mode": QUICK_MODE, "smoke_mode": SMOKE_MODE, "seed": SEED})

In [ ]:
def _rel(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)


def save_figure(fig, filename, dpi=300, write_pdf=None):
    if write_pdf is None:
        write_pdf = PAPER_FIGURE_MODE
    png_path = ch03.save_figure(fig, FIG_DIR, filename, dpi=dpi, write_pdf=write_pdf)
    rel_png = _rel(png_path)
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    if write_pdf:
        pdf_path = png_path.with_suffix(".pdf")
        rel_pdf = _rel(pdf_path)
        if rel_pdf not in paper_ready_pdf_written:
            paper_ready_pdf_written.append(rel_pdf)
    plt.close(fig)
    return png_path


def save_paper_figure(fig, stem, dpi=300):
    png_path = ch03.save_figure(fig, FIG_DIR, stem, dpi=dpi, write_pdf=True)
    pdf_path = png_path.with_suffix(".pdf")
    for path, bucket in [(png_path, paper_ready_png_written), (pdf_path, paper_ready_pdf_written)]:
        rel = _rel(path)
        if rel not in bucket:
            bucket.append(rel)
    rel_png = _rel(png_path)
    if rel_png not in figures_written:
        figures_written.append(rel_png)
    plt.close(fig)
    return png_path, pdf_path


def save_csv(frame, filename):
    path = ch03.save_csv(TABLE_DIR / filename, pd.DataFrame(frame))
    rel = _rel(path)
    if rel not in tables_written:
        tables_written.append(rel)
    return path


def save_paper_table(frame, stem, index=False):
    paths = ch03.save_paper_table(TABLE_DIR / stem, pd.DataFrame(frame), index=index)
    for path in paths:
        rel = _rel(path)
        if rel not in paper_tables_written:
            paper_tables_written.append(rel)
    return paths


def save_run_json(payload, filename):
    return ch03.save_json(OUT_DIR / filename, payload)


def display_saved_figure(path, width=None):
    return ch03.display_saved_figure(path, width=width)


def display_saved_figures(paths, width=None):
    return ch03.display_saved_figures(paths, width=width)


def display_table(frame, columns=None, n=10):
    return ch03.display_table(pd.DataFrame(frame), columns=columns, n=n)

## 1. 2D Toy Sanity Check

We begin with a deliberately low-dimensional problem because every object can be plotted: source samples, target samples, sampled endpoint pairs, and the learned vector field.

In [ ]:
def make_eight_gaussians(n=1500, radius=2.0, noise=0.08, seed=42):
    local_rng = np.random.default_rng(seed)
    angles = np.linspace(0.0, 2.0 * np.pi, 8, endpoint=False)
    centers = np.column_stack([radius * np.cos(angles), radius * np.sin(angles)])
    component = local_rng.integers(0, 8, size=n)
    points = centers[component] + local_rng.normal(scale=noise, size=(n, 2))
    return points.astype(np.float32), component


def make_single_gaussian(n=1500, loc=(0.0, 0.0), scale=(0.42, 0.28), angle=0.35, seed=43):
    local_rng = np.random.default_rng(seed)
    raw = local_rng.normal(size=(n, 2)).astype(np.float32)
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]], dtype=np.float32)
    S = np.diag(np.asarray(scale, dtype=np.float32))
    points = raw @ S @ R.T + np.asarray(loc, dtype=np.float32)
    return points.astype(np.float32)


def make_random_pair_batch_fn(X0, X1, seed=42):
    local_rng = np.random.default_rng(seed)

    def pair_batch_fn(batch_size):
        idx0 = local_rng.integers(0, len(X0), size=int(batch_size))
        idx1 = local_rng.integers(0, len(X1), size=int(batch_size))
        return {"x0": X0[idx0].astype(np.float32), "x1": X1[idx1].astype(np.float32)}

    return pair_batch_fn


toy_n = 1500 if QUICK_MODE else 5000
if SMOKE_MODE:
    toy_n = 350

toy_X0, toy_components = make_eight_gaussians(n=toy_n, seed=SEED)
toy_X1 = make_single_gaussian(n=toy_n, seed=SEED + 1)
toy_pair_batch_fn = make_random_pair_batch_fn(toy_X0, toy_X1, seed=SEED + 2)

toy_steps = 550 if QUICK_MODE else 2200
if SMOKE_MODE:
    toy_steps = 35
toy_log_every = 25 if not SMOKE_MODE else 5

toy_model = VelocityMLP(x_dim=2, hidden_dim=128, hidden_layers=4).to(DEVICE)
toy_optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)

In [ ]:
toy_history = train_cfm_steps(
    toy_model,
    toy_pair_batch_fn,
    toy_optimizer,
    n_steps=toy_steps,
    batch_size=256 if not SMOKE_MODE else 96,
    device=DEVICE,
    log_every=toy_log_every,
)

fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(toy_history["step"], toy_history["loss"], color="#2F6B5E", linewidth=1.6)
ax.set_xlabel("optimization step")
ax.set_ylabel("CFM local MSE")
ax.set_title("Toy CFM training loss")
ax.grid(axis="y", color="0.92", linewidth=0.7)
save_figure(fig, "fig_toy_loss.png")
toy_history.tail()

In [ ]:
display_saved_figure(FIG_DIR / "fig_toy_loss.png", width=620)

In [ ]:
display_table(toy_history.tail(6), n=6)

## 2. Sample the learned toy flow

After training, Euler sampling moves source particles through the learned marginal vector field. This is not yet a biological claim; it is a visual sanity check for the CFM mechanism.

In [ ]:
toy_model.eval()
toy_sample_n = min(900 if QUICK_MODE else 1800, len(toy_X0))
if SMOKE_MODE:
    toy_sample_n = min(220, len(toy_X0))
toy_sample_idx = subsample_idx(len(toy_X0), toy_sample_n, seed=SEED + 3)
toy_x0_t = torch.as_tensor(toy_X0[toy_sample_idx], dtype=torch.float32, device=DEVICE)
toy_euler_steps = 100 if not SMOKE_MODE else 25
with torch.no_grad():
    toy_final_t, toy_traj_t, toy_nfe = euler_sample(toy_model, toy_x0_t, n_steps=toy_euler_steps, return_traj=True)
toy_traj = as_np(toy_traj_t)

toy_times = [0.0, 0.25, 0.5, 0.75, 1.0]
toy_step_idx = np.round(np.asarray(toy_times) * (toy_traj.shape[0] - 1)).astype(int)
xlim, ylim = robust_limits(toy_X0, toy_X1, toy_traj.reshape(-1, 2), margin=0.10)
fig, axes = plt.subplots(1, 5, figsize=(12.3, 2.65), sharex=True, sharey=True)
for ax, tau, step in zip(axes, toy_times, toy_step_idx):
    pts = toy_traj[step]
    ax.scatter(toy_X1[subsample_idx(len(toy_X1), 450, seed=61), 0], toy_X1[subsample_idx(len(toy_X1), 450, seed=61), 1], s=5, color="0.72", alpha=0.20, linewidths=0)
    ax.scatter(pts[:, 0], pts[:, 1], s=6, color="#2F6B5E", alpha=0.52, linewidths=0)
    format_axis(ax, xlim, ylim, xlabel="toy x1", ylabel="toy x2", title=f"t={tau:.2f}")
    if ax is not axes[0]:
        ax.set_ylabel("")
fig.suptitle("Toy population evolution under Euler sampling")
save_figure(fig, "fig_toy_evolution.png")
print({"toy_euler_nfe": toy_nfe, "toy_traj_shape": toy_traj.shape})

In [ ]:
display_saved_figure(FIG_DIR / "fig_toy_evolution.png", width=900)

### Figure 3.2: Conditional Velocity Versus Marginal Velocity

A single endpoint pair gives one conditional straight-line velocity. Many endpoint pairs passing through the same local region give many possible conditional velocities. The neural velocity field learns their local average behavior as a marginal vector field estimate.

This cell compares raw conditional velocities around a point with the network's learned marginal vector.

In [ ]:
def plot_toy_conditional_vs_marginal(model, X0, X1, device="cpu", seed=42):
    local_rng = np.random.default_rng(seed)
    n_pairs = min(1800, len(X0), len(X1))
    i0 = local_rng.integers(0, len(X0), size=n_pairs)
    i1 = local_rng.integers(0, len(X1), size=n_pairs)
    x0 = X0[i0]
    x1 = X1[i1]
    t_value = 0.5
    x_t = (1.0 - t_value) * x0 + t_value * x1
    u_t = x1 - x0

    center = np.median(x_t, axis=0)
    dist = np.linalg.norm(x_t - center[None, :], axis=1)
    radius = np.quantile(dist, 0.18)
    local = np.flatnonzero(dist <= max(radius, 1e-6))
    if len(local) > 120:
        local = local[subsample_idx(len(local), 120, seed=seed + 1)]

    model.eval()
    with torch.no_grad():
        center_t = torch.as_tensor(center[None, :], dtype=torch.float32, device=device)
        time_t = torch.full((1, 1), t_value, dtype=torch.float32, device=device)
        pred = model(center_t, time_t).detach().cpu().numpy()[0]
    mean_conditional = u_t[local].mean(axis=0)

    xlim, ylim = robust_limits(X0, X1, x_t[local], margin=0.12)
    fig, ax = plt.subplots(figsize=(5.2, 4.5))
    ax.scatter(X0[subsample_idx(len(X0), 550, seed=71), 0], X0[subsample_idx(len(X0), 550, seed=71), 1], s=5, color="#4C78A8", alpha=0.18, linewidths=0, label="source")
    ax.scatter(X1[subsample_idx(len(X1), 550, seed=72), 0], X1[subsample_idx(len(X1), 550, seed=72), 1], s=5, color="#D1495B", alpha=0.18, linewidths=0, label="target")
    ax.scatter(x_t[local, 0], x_t[local, 1], s=11, color="0.55", alpha=0.35, linewidths=0)
    ax.quiver(x_t[local, 0], x_t[local, 1], u_t[local, 0], u_t[local, 1], angles="xy", scale_units="xy", scale=12, width=0.003, color="#9CC9C2", alpha=0.52, label="conditional velocities")
    ax.quiver([center[0]], [center[1]], [mean_conditional[0]], [mean_conditional[1]], angles="xy", scale_units="xy", scale=5, width=0.010, color="0.18", alpha=0.85, label="local conditional average")
    ax.quiver([center[0]], [center[1]], [pred[0]], [pred[1]], angles="xy", scale_units="xy", scale=5, width=0.013, color="#B9352B", alpha=0.95, label="network prediction")
    ax.scatter([center[0]], [center[1]], s=42, color="#B9352B", edgecolor="white", linewidth=0.7, zorder=5)
    format_axis(ax, xlim, ylim, xlabel="toy state 1", ylabel="toy state 2", title="Conditional velocities collapse to a marginal vector")
    ax.legend(frameon=False, loc="best")
    return fig, {"center": center, "network_velocity": pred, "mean_conditional_velocity": mean_conditional}

fig, toy_fig32_info = plot_toy_conditional_vs_marginal(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 4)
save_figure(fig, "fig03_02_conditional_vs_marginal_toy.png")
toy_fig32_info

In [ ]:
display_saved_figure(FIG_DIR / "fig03_02_conditional_vs_marginal_toy.png", width=620)

### Figure 3.3: CFM Object Hierarchy

This figure separates the objects in CFM: one endpoint path, a population of endpoint-conditioned paths, and the learned marginal vector field at `t=0.5`. It is still a 2D teaching toy, not the EB analysis state space.

The hierarchy panel ties together endpoint pairs, interpolation paths, and the learned vector field.

In [ ]:
def plot_toy_cfm_object_hierarchy(model, X0, X1, device="cpu", seed=42):
    local_rng = np.random.default_rng(seed)
    n_paths = min(90, len(X0), len(X1))
    i0 = local_rng.integers(0, len(X0), size=n_paths)
    i1 = local_rng.integers(0, len(X1), size=n_paths)
    x0 = X0[i0]
    x1 = X1[i1]
    xlim, ylim = robust_limits(X0, X1, x0, x1, margin=0.12)

    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.8), sharex=True, sharey=True)
    for ax in axes:
        ax.scatter(X0[subsample_idx(len(X0), 500, seed=81), 0], X0[subsample_idx(len(X0), 500, seed=81), 1], s=5, color="#4C78A8", alpha=0.14, linewidths=0)
        ax.scatter(X1[subsample_idx(len(X1), 500, seed=82), 0], X1[subsample_idx(len(X1), 500, seed=82), 1], s=5, color="#D1495B", alpha=0.14, linewidths=0)
        format_axis(ax, xlim, ylim, xlabel="toy state 1", ylabel="toy state 2")

    a, b = x0[0], x1[0]
    path = np.stack([(1.0 - t) * a + t * b for t in np.linspace(0, 1, 50)], axis=0)
    axes[0].plot(path[:, 0], path[:, 1], color="0.18", linewidth=1.6)
    axes[0].scatter([a[0], b[0]], [a[1], b[1]], s=40, color=["#4C78A8", "#D1495B"], edgecolor="white", linewidth=0.7, zorder=4)
    mid = 0.5 * (a + b)
    vel = b - a
    axes[0].quiver([mid[0]], [mid[1]], [vel[0]], [vel[1]], angles="xy", scale_units="xy", scale=4, width=0.012, color="#2F6B5E")
    axes[0].set_title("A. one endpoint path")

    for j in range(n_paths):
        axes[1].plot([x0[j, 0], x1[j, 0]], [x0[j, 1], x1[j, 1]], color="0.25", alpha=0.18, linewidth=0.65)
    axes[1].set_title("B. many endpoint paths")
    axes[1].set_ylabel("")

    grid_n = 20
    xs = np.linspace(xlim[0], xlim[1], grid_n)
    ys = np.linspace(ylim[0], ylim[1], grid_n)
    gx, gy = np.meshgrid(xs, ys)
    pts = np.column_stack([gx.ravel(), gy.ravel()]).astype(np.float32)
    model.eval()
    with torch.no_grad():
        x_t = torch.as_tensor(pts, dtype=torch.float32, device=device)
        t_t = torch.full((pts.shape[0], 1), 0.5, dtype=torch.float32, device=device)
        v = model(x_t, t_t).detach().cpu().numpy()
    norm = np.linalg.norm(v, axis=1)
    cap = np.percentile(norm[norm > 0], 88) if np.any(norm > 0) else 1.0
    v = v * np.minimum(1.0, cap / np.clip(norm[:, None], 1e-12, None))
    axes[2].quiver(pts[:, 0], pts[:, 1], v[:, 0], v[:, 1], angles="xy", scale_units="xy", scale=16, width=0.004, color="#B9352B", alpha=0.72)
    axes[2].set_title("C. learned marginal field at t=0.5")
    axes[2].set_ylabel("")
    fig.suptitle("CFM object hierarchy on a 2D toy problem")
    return fig

fig = plot_toy_cfm_object_hierarchy(toy_model, toy_X0, toy_X1, device=DEVICE, seed=SEED + 5)
save_figure(fig, "fig03_03_cfm_object_hierarchy_toy.png")

In [ ]:
display_saved_figure(FIG_DIR / "fig03_03_cfm_object_hierarchy_toy.png", width=900)

## Artifact audit

This notebook owns only the toy figures. EB training and all real-data diagnostics belong to the later splits.

In [ ]:
expected_figures = [
    FIG_DIR / "fig_toy_loss.png",
    FIG_DIR / "fig_toy_evolution.png",
    FIG_DIR / "fig03_02_conditional_vs_marginal_toy.png",
    FIG_DIR / "fig03_03_cfm_object_hierarchy_toy.png",
]
expected_tables = []
expected_outputs = [OUT_DIR / "run_config_03_1_toy_flow_matching_from_scratch.json"]
save_run_json({"seed": SEED, "quick_mode": QUICK_MODE, "smoke_mode": SMOKE_MODE, "owned_figures": [p.name for p in expected_figures]}, expected_outputs[0].name)
artifact_manifest = ch03.check_required_artifacts(expected_figures=expected_figures, expected_tables=expected_tables, expected_outputs=expected_outputs)
if not artifact_manifest["exists"].all():
    raise FileNotFoundError("Missing toy Flow Matching artifacts")
ch03.save_csv(OUT_DIR / "artifact_manifest_03_1_toy_flow_matching_from_scratch.csv", artifact_manifest)
display(artifact_manifest)